# Interpretability and SHAP

## Overview

Model interpretability explains why a model makes the predictions it does. This is essential for scientific credibility, regulatory requirements, and debugging.

**Interpretability methods:**

| Method | Scope | Model-agnostic |
|---|---|---|
| Coefficients | Global | No (linear models only) |
| Permutation importance | Global | Yes |
| Partial dependence plots (PDP) | Global | Yes |
| SHAP values | Global + local | Yes |
| LIME | Local | Yes |

SHAP (SHapley Additive exPlanations) is the gold standard: it decomposes each prediction into additive contributions from each feature, with theoretical guarantees from cooperative game theory.

**SHAP properties:** local accuracy, missingness, consistency — no other method satisfies all three simultaneously.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
n = 300
elevation   = rng.uniform(50, 400, n)
nitrate     = rng.gamma(2, 2, n)
phosphorus  = 0.6*nitrate + rng.normal(0, 0.5, n)
ph          = 7.5 - 0.003*elevation + rng.normal(0, 0.2, n)
treatment   = rng.choice([0,1], n)
richness    = (28 - 0.04*elevation - 0.9*nitrate + 2.5*treatment
               - 0.5*ph + rng.normal(0, 3, n))
feat_names  = ["elevation","nitrate","phosphorus","ph","treatment"]
X = np.column_stack([elevation, nitrate, phosphorus, ph, treatment])
X_tr, X_te, y_tr, y_te = train_test_split(X, richness, test_size=0.25, random_state=42)
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
print(f"RF R2 on test: {rf.score(X_te, y_te):.3f}")

---
## Permutation Importance

In [ ]:
perm = permutation_importance(rf, X_te, y_te, n_repeats=30,
                              random_state=42, n_jobs=-1)
pi_df = pd.DataFrame({
    "feature":    feat_names,
    "importance": perm.importances_mean,
    "std":        perm.importances_std
}).sort_values("importance", ascending=True)
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(pi_df["feature"], pi_df["importance"],
        xerr=pi_df["std"], color="steelblue", alpha=0.8,
        error_kw={"capsize":4})
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Mean accuracy decrease (permutation)")
ax.set_title("Permutation Importance on Test Set")
plt.tight_layout(); plt.show()
print("Features with importance near zero or negative: little predictive value")

---
## Partial Dependence Plots

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(14,4))
PartialDependenceDisplay.from_estimator(
    rf, X_te, features=[0,1,4],
    feature_names=feat_names,
    ax=axes, kind="both",   # "both" = PDP + individual conditional expectation
    subsample=50, random_state=42
)
axes[0].set_title("Elevation (PDP + ICE)")
axes[1].set_title("Nitrate (PDP + ICE)")
axes[2].set_title("Treatment (PDP + ICE)")
plt.suptitle("Partial Dependence: Marginal Effect of Each Feature", y=1.01)
plt.tight_layout(); plt.show()
print("PDP: average prediction across the dataset")
print("ICE: individual prediction path for each observation")
print("When ICE lines cross the PDP, there are interaction effects")

---
## SHAP Values

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_te)
    # Summary plot (beeswarm)
    plt.figure(figsize=(9,5))
    shap.summary_plot(shap_values, X_te,
                      feature_names=feat_names, show=False)
    plt.title("SHAP Summary Plot: Feature Impact on Model Output")
    plt.tight_layout(); plt.show()
    # Mean absolute SHAP (global importance)
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    print("Mean |SHAP| by feature:")
    for feat, val in sorted(zip(feat_names, mean_abs_shap),
                             key=lambda x: -x[1]):
        print(f"  {feat:12s}: {val:.4f}")
except ImportError:
    print("shap not installed: pip install shap")
    print("SHAP provides feature-level explanations for each prediction:")
    print("  shap.TreeExplainer(model).shap_values(X) -> (n_samples, n_features)")
    print("  Positive SHAP = pushed prediction up; Negative = pushed down")

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_te)
    # Local explanation: single prediction
    idx = 0
    pred = rf.predict(X_te[idx:idx+1])[0]
    print(f"Prediction for site {idx}: {pred:.2f}")
    print(f"Base value (mean prediction): {explainer.expected_value:.2f}")
    print("SHAP contributions:")
    for feat, sv in zip(feat_names, shap_values[idx]):
        direction = "up" if sv > 0 else "down"
        print(f"  {feat:12s}: {sv:+.4f}  (pushed prediction {direction})")
    # SHAP interaction: two-feature dependence
    fig, ax = plt.subplots(figsize=(7,5))
    sc = ax.scatter(X_te[:,0], shap_values[:,0],
                    c=X_te[:,1], cmap="viridis", s=20, alpha=0.7)
    plt.colorbar(sc, ax=ax, label="Nitrate")
    ax.set_xlabel("Elevation"); ax.set_ylabel("SHAP value for elevation")
    ax.set_title("SHAP Dependence Plot: Elevation effect, coloured by Nitrate")
    plt.tight_layout(); plt.show()
except ImportError:
    print("pip install shap to use SHAP waterfall and dependence plots")

---

## Common Pitfalls

**1. Using built-in RF/GBT impurity importance instead of permutation importance**  
Built-in impurity-based importance (mean decrease in impurity) is biased toward high-cardinality and continuous features — they get selected for splits more often by chance. Always use permutation importance or SHAP for reliable feature rankings.

**2. Reading PDP marginal effects when strong feature interactions exist**  
PDPs average over the marginal distribution of other features. When features interact strongly, the PDP misrepresents the true effect for specific subgroups. Always plot ICE curves alongside PDPs to check for interaction-driven heterogeneity.

**3. Interpreting SHAP values as causal effects**  
SHAP values decompose the prediction — they tell you how much each feature contributed to this model's output, not what would happen if you changed the feature value in the real world. Causal interpretation requires causal assumptions beyond what SHAP provides.

**4. Explaining a poorly predictive model**  
If the model has low predictive accuracy, its SHAP values explain a poor model — they describe the model's errors, not the real data-generating process. Always check model performance before investing in interpretability analysis.

**5. Reporting global SHAP importance without local explanations for decision-relevant cases**  
Global importance summaries can mask important heterogeneity. A feature with moderate global mean |SHAP| may dominate predictions for a specific subgroup. For decision-making, always examine SHAP values for the specific cases being acted upon.
---
*python_methods_library - Samantha McGarrigle*